#  Preprocesamiento de Imágenes Histopatológicas
## Proyecto: Predicción de Cáncer Bucal

Este notebook automatiza el pipeline de preparación de datos, realizando las siguientes tareas:
1. **Consolidación**: Ignora la estructura original y reúne todas las imágenes de `Normal` y `OSCC` de todas las carpetas del dataset original.
2. **Normalización**: Convierte todas las imágenes a modo **RGB** y las redimensiona a **224x224** píxeles usando interpolación Lanczos.
3. **Split Personalizado**: Divide el inventario total en conjuntos de **Entrenamiento (80%)**, **Validación (10%)** y **Prueba (10%)** de forma estratificada por clase.
4. **Exportación**: Almacena los resultados en una nueva estructura organizada bajo `data_procesada/`.

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
print(" Librerías cargadas correctamente.")

 Librerías cargadas correctamente.


--- 
## 1. Configuración de Parámetros

In [2]:
DATA_DIR = '../data'
OUTPUT_DIR = '../data_procesada'
IMG_SIZE = (224, 224)
RANDOM_STATE = 42

# Proporciones del split
TEST_SIZE = 0.1
VAL_SIZE = 0.1 # Del total original

CLASSES = ['Normal', 'OSCC']

--- 
## 2. Consolidación del Inventario de Imágenes
Buscamos recursivamente todas las imágenes en la carpeta `data` para crear un solo montón.

In [3]:
data_records = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            # Determinar clase basándonos en la ruta
            class_name = None
            for c in CLASSES:
                if c in root:
                    class_name = c
                    break
            
            if class_name:
                data_records.append({
                    'original_path': os.path.join(root, file),
                    'class': class_name,
                    'filename': file
                })

df = pd.DataFrame(data_records)
print(f" Total de imágenes encontradas: {len(df)}")
print("Distribución original:")
print(df['class'].value_counts())
print(df['class'].value_counts())
df.head()

 Total de imágenes encontradas: 5192
Distribución original:
class
OSCC      2698
Normal    2494
Name: count, dtype: int64
class
OSCC      2698
Normal    2494
Name: count, dtype: int64


,original_path,class,filename
0,../data\test\Normal\Normal_100x_12.jpg,Normal,Normal_100x_12.jpg
1,../data\test\Normal\Normal_100x_2.jpg,Normal,Normal_100x_2.jpg
2,../data\test\Normal\Normal_100x_21.jpg,Normal,Normal_100x_21.jpg
3,../data\test\Normal\Normal_100x_22.jpg,Normal,Normal_100x_22.jpg
4,../data\test\Normal\Normal_100x_25.jpg,Normal,Normal_100x_25.jpg


--- 
## 3. Split Estratificado (80/10/10)
Dividimos el dataset asegurando que las proporciones de `Normal` vs `OSCC` se mantengan en cada set.

In [4]:
# Primero separamos Train (80%) del resto (20%)
train_df, tmp_df = train_test_split(
    df, 
    test_size=(TEST_SIZE + VAL_SIZE), 
    stratify=df['class'], 
    random_state=RANDOM_STATE
)

# Del restante 20%, dividimos a la mitad para Val (10%) y Test (10%)
val_df, test_df = train_test_split(
    tmp_df, 
    test_size=0.5, 
    stratify=tmp_df['class'], 
    random_state=RANDOM_STATE
)

train_df['subset'] = 'train'
val_df['subset'] = 'val'
test_df['subset'] = 'test'

df_final = pd.concat([train_df, val_df, test_df])

print(" Split completado.")
print(df_final.groupby(['subset', 'class']).size().unstack())

 Split completado.
class   Normal  OSCC
subset              
test       250   270
train     1995  2158
val        249   270


--- 
## 4. Pipeline de Procesamiento
Definimos la función para procesar, redimensionar y guardar.

In [5]:
def process_and_save_image(row):
    try:
        # Crear ruta de destino
        dest_dir = os.path.join(OUTPUT_DIR, row['subset'], row['class'])
        os.makedirs(dest_dir, exist_ok=True)
        
        dest_path = os.path.join(dest_dir, row['filename'])
        
        # Procesar imagen
        with Image.open(row['original_path']) as img:
            # Convertir a RGB si es necesario
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Redimensionar
            img = img.resize(IMG_SIZE, Image.Resampling.LANCZOS)
            
            # Guardar
            img.save(dest_path, quality=95)
            
        return True
    except Exception as e:
        print(f" Error procesando {row['original_path']}: {e}")
        return False

print("⏳ Iniciando procesamiento...")
tqdm.pandas()
df_final['processed'] = df_final.progress_apply(process_and_save_image, axis=1)

print(f" Procesamiento terminado. Éxitos: {df_final['processed'].sum()} de {len(df_final)}")

⏳ Iniciando procesamiento...


  0%|          | 0/5192 [00:00<?, ?it/s]

 Procesamiento terminado. Éxitos: 5192 de 5192


--- 
## 5. Verificación Final

In [6]:
total_saved = 0
for subset in ['train', 'val', 'test']:
    for cls in CLASSES:
        path = os.path.join(OUTPUT_DIR, subset, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f" {subset}/{cls}: {count} imágenes")
            total_saved += count

print(f"\n Total global en carpeta procesada: {total_saved}")

 train/Normal: 1995 imágenes
 train/OSCC: 2158 imágenes
 val/Normal: 249 imágenes
 val/OSCC: 270 imágenes
 test/Normal: 250 imágenes
 test/OSCC: 270 imágenes

 Total global en carpeta procesada: 5192


--- 
## 6. Generación del Manifiesto
Guardamos un archivo CSV con las rutas relativas y etiquetas para facilitar el entrenamiento.

In [7]:
import pandas as pd
import os

# Crear rutas relativas para el entrenamiento
df_final['path_relativo'] = df_final.apply(lambda x: os.path.join(x['subset'], x['class'], x['filename']), axis=1)

# Asignar etiquetas numéricas
class_to_idx = {'Normal': 0, 'OSCC': 1}
df_final['label'] = df_final['class'].map(class_to_idx)

# Seleccionar columnas necesarias
manifest_df = df_final[['path_relativo', 'class', 'label', 'subset']]
manifest_df.rename(columns={'path_relativo': 'filepath'}, inplace=True)

# OUTPUT_DIR viene de celdas anteriores
manifest_path = os.path.join(r'../data_procesada', 'manifiesto.csv')
manifest_df.to_csv(manifest_path, index=False)

print(f" Manifiesto guardado en: {manifest_path}")
print(manifest_df.head())

 Manifiesto guardado en: ../data_procesada\manifiesto.csv
                           filepath   class  label subset
2120    train\Normal\aug_7_1850.jpg  Normal      0  train
2189  train\Normal\aug_851_7576.jpg  Normal      0  train
1419  train\Normal\aug_338_9737.jpg  Normal      0  train
2852    train\OSCC\aug_192_9707.jpg    OSCC      1  train
4844   train\OSCC\OSCC_400x_293.jpg    OSCC      1  train
